In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPooling2D, Flatten, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW
from tqdm.auto import tqdm
#from sklearn.metrics import confusion_matrix
#from sklearn.model_selection import training

In [ ]:
# crop image function

def crop_image(img):
    '''crops image into 300 x 300'''
    crop_box = (75, 75, 375, 375)
    cropped_img = img.crop(crop_box)
    cropped_img = img.resize((32,32)) #resize to 32 x 32
    cropped_img = img.convert('L') #returns a black and white image
    return cropped_img

In [ ]:
# array assignment function

def array_assignment(im):
    '''breaks image down to four dimensional input array and assigns ground truth label array'''
    #(width, height) = im.size
    (width, height) = (32,32)

    for x_pixel in range(width):
        for y_pixel in range(height):
            #(R,G,B) = im.getpixel((x_pixel,y_pixel))

            # for dimension 4, 0=R, 1=G, 2=B
            #x[image_index,x_pixel,y_pixel,0] = R
            #x[image_index,x_pixel,y_pixel,1] = G
            #x[image_index,x_pixel,y_pixel,2] = B

            x[image_index, x_pixel, y_pixel] = im.getpixel((x_pixel,y_pixel))

    # assigns 0 to image if it does not have leukemia
    if "hem" in str(file_path):
        y[image_index] = 0

    # assigns 1 to image if it has leukemia
    elif "all" in str(file_path):
        y[image_index] = 1

    else: 
        pass
            

In [ ]:
from pathlib import Path
from PIL import Image

#folder_path = Path("/Users/chloezeng/Documents/test png") #accesses folder with all our png images, CHLOES VERSION
folder_path = Path("C:/Users/piggi/Downloads/leukemia data/C-NMC_Leukemia/giant_input_folder") #SHANDIS VERSION
#png_files = folder_path.glob("*.png") #accesses individual file paths within our folder #CHLOES VERSION
bmp_files = folder_path.glob("*.bmp") #accesses individual file paths within our folder #SHANDIS VERSION

#x = np.empty((10661,300,300,3), dtype=np.uint8) #defines empty x (input) array
x = np.empty((10661,32,32), dtype=np.uint8) #FOR BLACK AND WHITE
y = np.empty((10661,), dtype=np.uint8) #defines empty y (ground truth/output) array

image_index = 0 #image number

for file_path in tqdm(bmp_files, desc="image processing progress:"): #shandi's version
    with Image.open(file_path) as img:
    #opens all images sequentially
        
        cropped_image = crop_image(img) #crops out extra black background
        
        array_assignment(cropped_image) #extracts data from pictures

        image_index += 1

#splits data into training, validation, and testing sets 80-10-10
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size = 0.2, random_state = 30)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size = 0.5, random_state = 67) 

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

In [ ]:
#testing if the code actually works

print("Current folder:", Path.cwd())

#folder_path = Path("/Users/chloezeng/Documents/test png") #chloes version
folder_path = Path("C:/Users/piggi/Downloads/leukemia data/C-NMC_Leukemia/giant_input_folder") #shandis version
print("Folder exists:", folder_path.exists())

#png_files = list(folder_path.glob("*.png")) #chloe's version
bmp_files = list(folder_path.glob("*.bmp")) #shandi's version
print("Number of files:", len(bmp_files)) #shandi's version

In [ ]:
# defining the CNN model

#model = Sequential()

#model.add(Input(shape=(300,300,3)))
#model.add(Conv2D(400, (3,3), activation='relu'))
#model.add(MaxPooling2D((2, 2)))
#model.add(Conv2D(800, (3, 3), activation='relu'))
#model.add(MaxPooling2D((2, 2)))
#model.add(Conv2D(1000, (3,3), activation = 'relu'))
##model.add(Flatten())
#model.add(GlobalAveragePooling2D())
#model.add(Dense(400, activation = 'relu'))
#model.add(Dense(200, activation = 'relu'))
#model.add(Dense(2, activation = 'softmax'))
#model.summary()

#smaller version
def create_model():
    model = Sequential()
    model.add(Input(shape=(32,32)))
    model.add(Conv2D(200, (3,3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(400, (3, 3), activation='relu'))
    #model.add(Flatten())
    model.add(GlobalAveragePooling2D())
    model.add(Dense(400, activation = 'relu'))
    model.add(Dense(2, activation = 'softmax'))
    model.summary()

    return model

# compiling the CNN model

learning_rates = [1e-2, 1e-3, 1e-4]
histories = {}

for lr in learning_rates:

    model = create_model()
    
    model.compile(optimizer = AdamW(learning_rate=lr), 
                  loss = 'binary_crossentropy', 
                  metrics = ['accuracy'])
    #model.compile(optimizer = AdamW(learning_rate=1e-3), loss = 'binary_crossentropy', metrics = ['accuracy'])
    
    # Train the model
    
    history = model.fit(x_train, y_train, 
                        epochs = 5, 
                        batch_size = 8, 
                        validation_data = (x_val, y_val))

    histories[lr] = history

    plt.figure(figsize=(8,5))

for lr, history in histories.items():
    plt.plot(history.history['loss'], label = f'Learning rate = {lr}')

plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('training loss for different learning rates')
plt.legend()
plt.show()

#evaluating accuracy

y_pred = model.predict(x_val)
y_pred = tf.argmax(y_pred, axis = 1)
loss, acc = model.evaluate(x_val, y_val)
print ("validation accuracy:", acc)